In [ ]:
import json
import re
import pandas as pd

# ==========================================
# KHỐI 0: CÁC HÀM TIỆN ÍCH CƠ BẢN
# ==========================================
def load_dict_from_txt(filepath):
    """Đọc file txt dạng list thông thường (1 từ 1 dòng)"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return [line.strip().lower() for line in f if line.strip()]
    except FileNotFoundError:
        print(f"[CẢNH BÁO] Không tìm thấy file {filepath}.")
        return []

def load_skill_mapping(filepath):
    """Đọc file txt chứa Vecto Q dạng (Key: val1, val2)"""
    skill_dict = {}
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or ':' not in line:
                    continue
                # Tách Tên_Kỹ_năng và Chuỗi_Từ_khóa
                key, values_str = line.split(':', 1)
                # Tách các từ khóa bằng dấu phẩy và xóa khoảng trắng thừa
                keywords = [kw.strip().lower() for kw in values_str.split(',')]
                skill_dict[key.strip()] = keywords
    except FileNotFoundError:
        print(f"[CẢNH BÁO] Không tìm thấy file {filepath}.")
    return skill_dict

def clean_text(text):
    if not text: return ""
    return str(text).replace('$$', ' ').replace('$', ' ').lower()

# TẢI TOÀN BỘ TỪ ĐIỂN TỪ BÊN NGOÀI
DICT_VOCAB = load_dict_from_txt("D:\\IntelligentTesting\\IntelligentTesting\\Dictionary\\I_vocab_difficulty.txt")
DICT_SYNTACTIC = load_dict_from_txt("D:\\IntelligentTesting\\IntelligentTesting\\Dictionary\\I_syntactic_complexity.txt")
DICT_SYMBOLS = load_dict_from_txt("D:\\IntelligentTesting\\IntelligentTesting\\Dictionary\\II_operator_count.txt")
SKILL_MAPPING = load_skill_mapping("D:\\IntelligentTesting\\IntelligentTesting\\Dictionary\\Q_vecto.txt") # Tải Vecto Q

# ==========================================
# KHỐI 1: MODULE TRÍCH XUẤT ĐẶC TRƯNG NGÔN NGỮ (Nhóm I)
# ==========================================
def extract_linguistic_features(content_text):
    text = clean_text(content_text)
    words = re.findall(r'\b\w+\b', text)
    word_count = max(len(words), 1)
    
    # 1. Độ dài từ & câu
    avg_word_length = sum(len(w) for w in words) / word_count
    sentences = [s for s in re.split(r'[.!?]+(?=\s|$)', text) if s.strip()]
    sentence_count = max(len(sentences), 1)
    avg_sentence_length = word_count / sentence_count
    
    # 2. Quét từ điển Từ vựng khó
    vocab_count = sum(text.count(term) for term in DICT_VOCAB)
    vocab_difficulty = vocab_count / word_count
    
    # 3. Quét từ điển Cú pháp
    syntactic_count = sum(text.count(term) for term in DICT_SYNTACTIC)
    syntactic_complexity = syntactic_count / sentence_count
    
    return {
        "I_Word_Count": word_count,
        "I_Avg_Word_Length": round(avg_word_length, 2),
        "I_Avg_Sentence_Length": round(avg_sentence_length, 2),
        "I_Vocab_Difficulty": round(vocab_difficulty, 3),
        "I_Syntactic_Complexity": round(syntactic_complexity, 3)
    }

# ==========================================
# KHỐI 2: MODULE TRÍCH XUẤT MIỀN TOÁN HỌC (Nhóm II)
# ==========================================
def extract_domain_features(content_text):
    text = clean_text(content_text)
    words = re.findall(r'\b\w+\b', text)
    word_count = max(len(words), 1)
    
    # 1. N_s (Symbol): Đếm số lượng Ký hiệu / Toán tử / Đơn vị
    # Chỉ đếm khi ký hiệu đứng độc lập hoặc đi kèm số (tránh đếm chữ 'm' trong chữ 'Năm')
    N_s = 0
    for sym in DICT_SYMBOLS:
        # Nếu là ký hiệu toán học đặc biệt (+, -, *, /) thì dùng count bình thường
        if sym in ['+', '-', '=', '/', '%', '$']:
            N_s += text.count(sym)
        # Nếu là chữ cái (m, km, kg), bắt buộc phải dùng ranh giới từ \b
        else:
            pattern = r'\b' + re.escape(sym) + r'\b'
            N_s += len(re.findall(pattern, text))
    
    # 2. N_c (Concrete): Đếm số lượng dữ kiện cụ thể (Con số)
    concrete_matches = re.findall(r'\b\d+[\.,]?\d*\b', text)
    N_c = len(concrete_matches)
    
    # 3. N_a (Abstract): Đếm số lượng thuật ngữ trừu tượng 
    # (Sử dụng chung DICT_VOCAB từ Khối 0)
    N_a = sum(text.count(term) for term in DICT_VOCAB)
    
    # 4. Tính tổng thực thể có ý nghĩa (T)
    T = N_c + N_s + N_a
    
    # 5. Tính Phổ trừu tượng (Xử lý lỗi chia cho 0 nếu câu hỏi không có thực thể nào)
    if T > 0:
        P_concrete = N_c / T
        P_symbol = N_s / T
        P_abstract = N_a / T
    else:
        P_concrete = P_symbol = P_abstract = 0.0
        
    return {
        # Bổ sung 3 chiều của Phổ trừu tượng vào vector đầu ra
        "II_P_Concrete": round(P_concrete, 3),
        "II_P_Symbol": round(P_symbol, 3),
        "II_P_Abstract": round(P_abstract, 3)
    }

# ==========================================
# KHỐI 3: MODULE TRÍCH XUẤT ĐẶC TRƯNG ẨN & VECTO Q
# ==========================================
def extract_latent_and_q_vector(analysis_text, kc_routes_list):
    text = clean_text(analysis_text)
    
    # --- PHƯƠNG PHÁP CẢI TIẾN: ĐẾM PHÂN ĐOẠN TƯ DUY ---
    
    # 1. Tách lời giải thành các câu riêng biệt
    # Sử dụng Regex để tránh bị ngắt bởi số thập phân (như đã thảo luận)
    steps_raw = re.split(r'[.!?]+(?=\s|$)', text)
    # Lọc bỏ các khoảng trắng thừa
    steps = [s.strip() for s in steps_raw if s.strip()]
    
    # 2. Đếm số bước suy luận
    # Mỗi câu trong lời giải thường đại diện cho 1 bước tư duy logic hoặc 1 phép tính
    inference_steps = len(steps)
    
    # (Tùy chọn nâng cao) Nếu một câu có nhiều dấu '=', ta cộng thêm bước
    # vì học sinh thực hiện nhiều phép tính gộp trong 1 câu.
    for s in steps:
        extra_equals = s.count('=')
        if extra_equals > 1:
            inference_steps += (extra_equals - 1)

    kc_depth = len(str(kc_routes_list).split('----'))

    # 3. Trích xuất Vecto Q (Giữ nguyên logic cũ)
    routes_str = str(kc_routes_list).lower()
    vector_q = {}
    for q_key, keywords in SKILL_MAPPING.items():
        vector_q[q_key] = 1 if any(kw in routes_str for kw in keywords) else 0

    return {
        "III_Inference_Steps": inference_steps,
        "III_KC_Depth": kc_depth,
        **vector_q
    }

# ==========================================
# KHỐI 4: ORCHESTRATOR (HỢP NHẤT VÀ CHẠY TOÀN BỘ)
# ==========================================
def build_feature_dataset(json_filepath, output_csv):
    print(f"Đang đọc dữ liệu từ: {json_filepath}")
    with open(json_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    dataset = []
    total_items = len(data)
    print(f"Bắt đầu trích xuất đặc trưng cho {total_items} câu hỏi...")

    for item_id, item_data in data.items():
        content = item_data.get("content", "")
        analysis = item_data.get("analysis", "")
        kc_routes = item_data.get("kc_routes", [])

        # KHỞI TẠO DICT KẾT QUẢ CHO 1 CÂU HỎI
        item_features = {"ID": item_id}
        
        # GỌI TỪNG MODULE VÀ CẬP NHẬT VÀO KẾT QUẢ
        item_features.update(extract_linguistic_features(content))
        item_features.update(extract_domain_features(content))
        item_features.update(extract_latent_and_q_vector(analysis, kc_routes))
        
        # Thêm vào mảng tổng
        dataset.append(item_features)

    # Xuất ra DataFrame và lưu CSV
    df = pd.DataFrame(dataset)
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"✅ Hoàn tất! Đã lưu file kết quả tại: {output_csv}")
    
    return df

# ==========================================
# CHẠY CHƯƠNG TRÌNH
# ==========================================
if __name__ == "__main__":
    INPUT_FILE = "D:\\IntelligentTesting\\IntelligentTesting\\XES3G5M\\XES3G5M\\metadata\\question_full.json" # Đổi tên file JSON đầu vào của bạn tại đây
    OUTPUT_FILE = "full_features.csv"
    
    # Chạy hợp nhất
    df_result = build_feature_dataset(INPUT_FILE, OUTPUT_FILE)
    
    # In thử một vài cột để kiểm tra
    print("\n--- XEM TRƯỚC VÀI DÒNG KẾT QUẢ ---")
    print(df_result.head(2))

Đang đọc dữ liệu từ: D:\IntelligentTesting\IntelligentTesting\XES3G5M\XES3G5M\metadata\question_full.json
Bắt đầu trích xuất đặc trưng cho 7172 câu hỏi...
✅ Hoàn tất! Đã lưu file kết quả tại: full_features.csv

--- XEM TRƯỚC VÀI DÒNG KẾT QUẢ ---
  ID  I_Word_Count  I_Avg_Word_Length  I_Avg_Sentence_Length  \
0  0            56               3.25                   28.0   
1  1            45               3.40                   22.5   

   I_Vocab_Difficulty  I_Syntactic_Complexity  II_P_Concrete  II_P_Symbol  \
0                 0.0                     3.0            1.0          0.0   
1                 0.0                     3.0            0.8          0.2   

   II_P_Abstract  III_Inference_Steps  Q1_TinhToan  Q2_LyThuyetSo  Q3_HinhHoc  \
0            0.0                    4            0              0           0   
1            0.0                    4            0              0           0   

   Q4_ChuyenDong  Q5_ToanDoKinhDien  Q6_TongHieuTi  Q7_Dem_ToHop  \
0              0 